# Приложение Б к Модулю 11.5. Полноценный смолагент локально — без единого токена

«Полноценный смолагент» — это не про облако. Агенту нужны четыре вещи: мир, инструменты, движок и модель — и все четыре могут целиком жить на вашей машине. В этом приложении лесоруб из основного ноутбука работает с **локальной** моделью: ни одного ключа или токена не нужно вовсе.

Это развёрнутая версия Блока 3 основного ноутбука: там локальный прогон дан одной компактной ячейкой, здесь тот же маршрут пройден не спеша — как найти сервер, как выбрать модель и что может пойти не так. Мир `Forest`, четыре инструмента и `ToolCallingAgent` — те же самые, дословно.

Локальный сервер модели вы уже поднимали в курсе: [Модуль 6.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/06-1-local-first/) — Ollama, llama.cpp, vLLM — и [Модуль 6.2](https://itrubnikov.github.io/Train_of_Thought/docs/modules/06-2-local-llm/) — LM Studio. Подойдёт любой: оба поднимают один и тот же OpenAI-совместимый сервер.

Главное про запуск: ноутбук честно проходит целиком (`Run all`) и **без** запущенного локального сервера — ячейки напечатают причину пропуска, и это норма. Интернет, как и в основном ноутбуке, нужен один раз — поставить библиотеки (первая ячейка). Живой прогон случится сам, как только на машине найдётся Ollama или LM Studio с моделью.

In [ ]:
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import smolagents  # noqa: F401
except Exception:
    _pip_install("smolagents")
    import smolagents  # noqa: F401

try:
    import pydantic  # noqa: F401
except Exception:
    _pip_install("pydantic")
    import pydantic  # noqa: F401

try:
    import openai  # noqa: F401  # нужен для OpenAIServerModel
except Exception:
    _pip_install("openai")
    import openai  # noqa: F401

print("smolagents", smolagents.__version__,
      "| pydantic", pydantic.VERSION, "| openai", openai.__version__)

## Мир и инструменты — те же самые

Ниже — дословная копия из основного ноутбука: мок-лес `Forest`, карта `get_map` и финальный «хороший» набор — `MoveTool`, `GatherTool`, `deposit` — с `enum` в схеме, обучающими ошибками `{code, message}` и конвертом `{result, cooldown, state}`. Ни одной правки.

Это и есть главный тезис приложения: перед вами **те же объекты**, и контракту всё равно, где живёт модель. Эти четыре инструмента читал скриптовый харнесс Блока 2 и живая модель Блока 3 — сейчас вы разберёте локальный маршрут по шагам.

In [ ]:
import time


class Forest:
    """Мок-лес: сетка 5x5, лесоруб, рюкзак, склад и кулдаун."""

    COOLDOWN = 0.3  # в лекции 1.0; здесь меньше, чтобы Run all занимал ~минуту, — правила те же

    def __init__(self):
        self.size = 5
        self.nodes = {(1, 2): "wood", (3, 1): "wood", (2, 4): "stone"}
        self.pos = (0, 0)        # где стоит лесоруб
        self.home = (0, 0)       # клетка склада
        self.backpack = {}       # например, {"wood": 3}
        self.cap = 5             # вместимость рюкзака
        self.stock = {}          # что уже сдано на склад
        self.busy_until = 0.0    # когда закончится кулдаун

    def cooldown_left(self):
        return max(0.0, self.busy_until - time.monotonic())

    def start_cooldown(self):
        self.busy_until = time.monotonic() + self.COOLDOWN

    def backpack_load(self):
        return sum(self.backpack.values())


forest = Forest()


def reset_forest():
    """Свежий мир перед каждым сценарием. Tools ниже смотрят на глобальную forest."""
    global forest
    forest = Forest()


print("Лес собран:", forest.size, "x", forest.size,
      "| узлы:", forest.nodes, "| лесоруб и склад на", forest.home)


import json

from smolagents import tool, Tool


@tool
def get_map() -> dict:
    """Карта леса: позиция лесоруба, узлы ресурсов и клетка склада. Мир не меняет."""
    return {"pos": list(forest.pos), "home": list(forest.home),
            "nodes": [{"pos": list(p), "resource": r} for p, r in forest.nodes.items()]}


class GatherTool(Tool):
    name = "gather"
    description = (
        "Добыть один ресурс с клетки, на которой стоит лесоруб. "
        "Без resource берёт то, что есть на клетке; с resource — только ожидаемое."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood", "stone"],
            "nullable": True,
            "description": "Ожидаемый ресурс: wood или stone. По умолчанию — любой.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        node = forest.nodes.get(forest.pos)
        if node is None or (resource is not None and node != resource):
            return {"error": {"code": "no_resource_here",
                              "message": "На этой клетке нет нужного узла — "
                                         "найдите его через get_map и подойдите move."}}
        if forest.backpack_load() >= forest.cap:
            return {"error": {"code": "inventory_full",
                              "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — "
                                         "вернитесь на склад (0, 0) и позовите deposit."}}
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"result": {"gathered": node, "amount": 1},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


class MoveTool(Tool):
    name = "move"
    description = (
        "Шаг на одну клетку в сторону direction. "
        "Зовите, когда до нужного узла или склада не хватает шага."
    )
    inputs = {
        "direction": {
            "type": "string",
            "enum": ["north", "south", "east", "west"],
            "description": "Куда шагнуть: north, south, east или west.",
        }
    }
    output_type = "object"

    def forward(self, direction: str) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        dx, dy = {"north": (0, -1), "south": (0, 1),
                  "east": (1, 0), "west": (-1, 0)}[direction]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        forest.start_cooldown()
        return {"result": {"pos": list(forest.pos)},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


@tool
def deposit() -> dict:
    """Сдать содержимое рюкзака на склад. Работает только на клетке склада (0, 0)."""
    wait = forest.cooldown_left()
    if wait > 0:
        return {"error": {"code": "on_cooldown",
                          "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке (0, 0), а вы — на {forest.pos}. "
                                     "Дойдите до склада и повторите deposit."}}
    banked, forest.backpack = forest.backpack, {}
    for res, n in banked.items():
        forest.stock[res] = forest.stock.get(res, 0) + n
    forest.start_cooldown()
    return {"result": {"banked": banked},
            "cooldown": forest.COOLDOWN,
            "state": {"pos": list(forest.pos), "backpack": {}, "stock": forest.stock}}


move, gather = MoveTool(), GatherTool()
print("Инструменты собраны: move, gather, deposit, get_map")
print()
print("Контракт move глазами модели (enum прямо в схеме):")
print(json.dumps(move.inputs, ensure_ascii=False, indent=2))
print()
print("Контракт gather (enum + nullable):")
print(json.dumps(gather.inputs, ensure_ascii=False, indent=2))

## Ищем локальный сервер

Оба героя Модулей 6.1–6.2 поднимают OpenAI-совместимый сервер: Ollama — на `http://localhost:11434/v1`, LM Studio — на `http://localhost:1234/v1`. Проверим оба адреса запросом `GET /models` с коротким таймаутом — тем же приёмом, что ping живого API в Блоке 4 основного ноутбука.

Если ваш сервер живёт по другому адресу (например, vLLM), задайте переменные окружения `LOCAL_API_BASE` (скажем, `http://localhost:8000/v1`) и `LOCAL_MODEL_ID` — ячейка проверит их первыми. Модель выбирается так: `LOCAL_MODEL_ID`, если задан; иначе `qwen2.5:7b`, если есть в списке сервера; иначе первая модель из списка.

In [ ]:
import os

import requests

CANDIDATES = [
    ("Ollama", "http://localhost:11434/v1"),
    ("LM Studio", "http://localhost:1234/v1"),
]
if os.environ.get("LOCAL_API_BASE"):
    CANDIDATES.insert(0, ("LOCAL_API_BASE", os.environ["LOCAL_API_BASE"]))

LOCAL_BASE = None    # адрес найденного сервера
LOCAL_MODEL = None   # выбранная модель

for server_name, base in CANDIDATES:
    try:
        r = requests.get(f"{base}/models", timeout=2)
        if r.status_code != 200:
            print(f"{server_name}: {base} ответил {r.status_code} -> пропускаю.")
            continue
        model_ids = [m.get("id", "") for m in r.json().get("data", [])]
        print(f"{server_name}: сервер найден на {base}, моделей в списке: {len(model_ids)}")
        if os.environ.get("LOCAL_MODEL_ID"):
            LOCAL_MODEL = os.environ["LOCAL_MODEL_ID"]
            print("Модель из LOCAL_MODEL_ID:", LOCAL_MODEL)
        elif "qwen2.5:7b" in model_ids:
            LOCAL_MODEL = "qwen2.5:7b"
            print("Модель: qwen2.5:7b — есть в списке, берём её.")
        elif model_ids:
            LOCAL_MODEL = model_ids[0]
            print("qwen2.5:7b в списке нет -> берём первую доступную:", LOCAL_MODEL)
        else:
            print("Сервер есть, но моделей в нём нет — загрузите модель и перезапустите ячейку.")
            continue
        LOCAL_BASE = base
        break
    except Exception:
        print(f"{server_name}: {base} не отвечает.")

if LOCAL_BASE is None:
    print()
    print("Локальный сервер не найден -> живой прогон ниже будет пропущен (это норма).")
    print("Как поднять за несколько минут (подробности — в Модулях 6.1/6.2):")
    print("  Ollama:    установите с ollama.com (или: brew install ollama),")
    print("             затем: ollama pull qwen2.5:7b")
    print("             приложение само поднимет сервер на порту 11434.")
    print("  LM Studio: скачайте модель в GUI и включите Start Server (порт 1234).")

## Живой прогон: та самая одна строка

В Модуле 10 модель была облачной: `model = InferenceClientModel()`. Здесь — `model = OpenAIServerModel(model_id=..., api_base=...)`: одна строка выбирает, где живёт модель. `OpenAIServerModel` умеет говорить с любым OpenAI-совместимым сервером, а наш локальный — ровно такой.

`api_key="local"` — заглушка: локальный сервер ключ не проверяет, но поле обязательно. Задача лесорубу — та же, что в Блоке 3, дословно. Если сервер не найден или модель споткнётся, ячейка напечатает причину и мягко пропустит прогон — `Run all` останется зелёным.

In [ ]:
if LOCAL_BASE and LOCAL_MODEL:
    try:
        from smolagents import OpenAIServerModel, ToolCallingAgent

        reset_forest()
        model = OpenAIServerModel(
            model_id=LOCAL_MODEL,
            api_base=LOCAL_BASE,
            api_key="local",   # заглушка: локальный сервер ключ не проверяет
        )
        agent = ToolCallingAgent(
            tools=[move, gather, deposit, get_map],
            model=model,
            max_steps=12,
        )
        agent.run("Добудь одно дерево (wood) в лесу и сдай его на склад. "
                  "Действуй только инструментами; если получил ошибку — читай её code и message.")
        print()
        print("Склад после прогона:", forest.stock)
    except Exception as e:
        print("Живой прогон не прошёл -> мягкий пропуск:", repr(e))
else:
    print("Живой прогон пропущен: локальный сервер не найден. Это ожидаемо при офлайн-прогоне.")

## Оговорки про локальные модели

- **Нужен tool calling.** Берите instruct-модель размером от ~7B — `qwen2.5:7b` подходит и в Ollama, и в LM Studio. Совсем маленькие и не-instruct модели вызовам инструментов толком не обучены.
- **Модель поменьше будет чаще промахиваться мимо схемы** — перепутает имя аргумента, выдумает значение вне `enum`. И именно здесь обучающие ошибки `{code, message}` из Блока 1 основного ноутбука отрабатывают на полную: каждый промах возвращается модели понятным текстом и конвертируется в следующий осмысленный ход, а не в тупик.
- **Кулдаун 0.3 с локально почти не мешает:** пауза между шагами агента — пока модель думает над следующим вызовом — обычно длиннее. А если модель всё же поспешит, получит `on_cooldown` и повторит.
- **Хотите режим Модуля 10** — замените `ToolCallingAgent` на `CodeAgent`: модель будет писать вызовы кодом, а контракт останется тем же.

## Что дальше

`OpenAIServerModel` открывает дверь не только на localhost: сюда же встаёт любая OpenAI-совместимая точка — vLLM из Модуля 6.1, шлюзы вроде OpenRouter из Модуля 6.3. Меняется всё та же одна строка `model = ...`.

Соседний ноутбук-приложение `to-hf-space.ipynb` показывает другой маршрут: тот же агент — веб-чатом на HF Spaces, чтобы дать ссылку друзьям. А Блок 4 основного ноутбука — третий: те же принципы против живого API игры.